<a href="https://colab.research.google.com/github/rayychandana/PlantDoc-ResNet50-Classifier/blob/main/PlantDocPlantDiseaseDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Download the dataset directly from GitHub
!git clone https://github.com/pratikkayal/PlantDoc-Dataset.git

import os
print("Dataset successfully downloaded from GitHub!")

# 2. Set your base directory to the newly cloned folder
base_dir = '/content/PlantDoc-Dataset'
train_dir = os.path.join(base_dir, 'train')
test_dir = os.path.join(base_dir, 'test')

if os.path.exists(train_dir):
    print("Found Train folder!")
    print(f"Total classes in Train: {len(os.listdir(train_dir))}")
else:
    print("Path is incorrect. Please check the folder names again.")

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import preprocess_input

# Standardizing images to 224x224 for ResNet
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=40,
    width_shift_range=0.3,
    height_shift_range=0.3,
    shear_range=0.2,
    zoom_range=0.3,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.7, 1.3],
    fill_mode='reflect'
)

# Test generator should NOT have augmentation, only preprocessing
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

# === THE FIX ===
# Extract the exact list of 28 folders found in the training set
class_names = list(train_generator.class_indices.keys())

val_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=class_names, # Force the validation set to use the exact same 28 classes!
    shuffle=False
)

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras import regularizers

# Load ResNet50 without the top layer
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False # Freeze base model

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(0.01)),
    layers.BatchNormalization(),
    layers.Dropout(0.6),
    layers.Dense(train_generator.num_classes, activation='softmax')
])

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Stage 1: Warming up the head...")
history_stage1 = model.fit(train_generator, validation_data=val_generator, epochs=5)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import json

print("Stage 2: Surgical fine-tuning...")

# ==========================================
# Mount Drive right before the heavy training starts
# so your model saves permanently!
from google.colab import drive
drive.mount('/content/drive')
# ==========================================

base_model.trainable = True
for layer in base_model.layers[:-50]: # Unfreeze top 50 layers
    layer.trainable = False

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint(filepath='/content/drive/MyDrive/best_plant_model.keras',
                    monitor='val_accuracy',
                    save_best_only=True,
                    verbose=1)
]

history_stage2 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=callbacks
)

# Save the full model (Architecture + Weights + Optimizer state)
model.save('/content/drive/MyDrive/PlantDoc_ResNet50_Final.keras')

# Save the class mapping (Crucial for interpreting results later)
class_indices = train_generator.class_indices
with open('/content/drive/MyDrive/class_indices.json', 'w') as f:
    json.dump(class_indices, f)

print("Model and Class Indices saved to Drive successfully!")

In [ ]:
import tensorflow as tf
import json

# 1. Load the model from your Drive
loaded_model = tf.keras.models.load_model('/content/drive/MyDrive/PlantDoc_ResNet50_Final.keras')

# 2. Load the class indices to translate numbers to disease names
with open('/content/drive/MyDrive/class_indices.json', 'r') as f:
    class_indices = json.load(f)

# Create a reverse mapping (Index -> Name)
labels = {v: k for k, v in class_indices.items()}

print("Model loaded and ready for prediction!")

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Get predictions for the entire validation set
val_generator.shuffle = False
val_generator.reset()
Y_pred = loaded_model.predict(val_generator)
y_pred = np.argmax(Y_pred, axis=1)

# 2. Get the true labels and the class names
y_true = val_generator.classes
class_labels = list(val_generator.class_indices.keys())

# === THE FIX ===
# Create an array of expected labels [0, 1, 2, ..., 27]
expected_labels = np.arange(len(class_labels))

# 3. Print Classification Report
print("\n--- Classification Report ---")
# Add the 'labels' parameter and 'zero_division=0' to prevent warnings for the empty class
print(classification_report(
    y_true,
    y_pred,
    labels=expected_labels,
    target_names=class_labels,
    zero_division=0
))

# 4. Plot Confusion Matrix
plt.figure(figsize=(15, 12))
# Add the 'labels' parameter here too so the matrix size stays 28x28
cm = confusion_matrix(y_true, y_pred, labels=expected_labels)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels)
plt.title('Confusion Matrix')
plt.ylabel('Actual Disease')
plt.xlabel('Predicted Disease')
plt.xticks(rotation=90)
plt.show()

In [ ]:
def plot_actual_vs_predicted_clean(generator, model, num_images=9):
    # Reset generator to get a fresh batch
    generator.reset()
    images, labels_true = next(generator)

    # Get predictions for this batch
    preds = model.predict(images)

    plt.figure(figsize=(12, 12))

    # Make sure we don't try to plot more images than we have in a batch
    for i in range(min(num_images, len(images))):
        plt.subplot(3, 3, i+1)

        # Normalize image back to 0-1 range just for plotting
        img_to_show = images[i]
        img_to_show = (img_to_show - img_to_show.min()) / (img_to_show.max() - img_to_show.min())

        plt.imshow(img_to_show)

        actual_idx = np.argmax(labels_true[i])
        pred_idx = np.argmax(preds[i])

        # Use the 'labels' dictionary to get the string names
        actual_name = labels[actual_idx]
        pred_name = labels[pred_idx]

        color = 'green' if actual_idx == pred_idx else 'red'

        plt.title(f"Actual: {actual_name}\nPredicted: {pred_name}", color=color, fontsize=10)
        plt.axis('off')

    plt.tight_layout()
    plt.show()

plot_actual_vs_predicted_clean(val_generator, loaded_model)